# IndiVoice-DeepASR: Cloud Gateway 🚀

This notebook is your command center for training the **IndiVoice-DeepASR** model using Google Colab GPUs.

### 📁 Cloud-to-Cloud Workflow
This notebook is configured to download datasets directly from **Hugging Face** and **GitHub** into your Google Drive for persistent storage.

## 1. Environment & Drive Initialization

In [6]:
# 1. Verify GPU Availability
import torch
!nvidia-smi
if not torch.cuda.is_available():
    print("\n⚠️ WARNING: GPU is not enabled! Training will be extremely slow.")
    print("👉 Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
else:
    print(f"\n✅ GPU Detected: {torch.cuda.get_device_name(0)}")

# 2. Mount Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# 3. Configure Paths
DRIVE_BASE = '/content/drive/MyDrive/IndiVoice-DeepASR'
os.makedirs(DRIVE_BASE, exist_ok=True)

# 4. Sync Repository (Force Pull latest fixes)
%cd "{DRIVE_BASE}"
if not os.path.exists('.git'):
    !git clone https://github.com/purvanshjoshi/IndiVoice-DeepASR.git .
else:
    print("Hard Syncing Repo to pick up latest fixes...")
    !git fetch --all
    !git clean -fd
    !git reset --hard origin/main
    !git checkout -f main

# 5. Install Dependencies
!pip install -r requirements.txt

print("\n✅ Setup Complete. All scripts (src/) are verified in Drive.")

Sat Mar 14 15:42:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Hugging Face Authentication (Required for Svarah)
The Svarah dataset is **gated**. Follow these steps first:
1. Visit [huggingface.co/datasets/ai4bharat/Svarah](https://huggingface.co/datasets/ai4bharat/Svarah) and click **'Agree and access repository'** (you'll need a free HF account).
2. Get your Access Token from [hf.co/settings/tokens](https://hf.co/settings/tokens).
3. Run the cell below and paste your token.

In [7]:
from huggingface_hub import login
login()

## 3. Cloud Data Acquisition
Run this cell to download the research datasets (Svarah, IndicAccentDb, etc.) directly into your Google Drive.

In [8]:
# Downloads datasets directly to data/raw in your Drive
!python src/download_data.py --data_dir data/raw

🚀 Downloading Svarah Dataset from Hugging Face...
✅ Svarah Loaded/Cached from Hugging Face (using 'test' split).
🚀 Downloading IndicAccentDb from Hugging Face...
✅ IndicAccentDb loaded/cached.
⚠️ WARNING: NPTEL2020 is extremely large (15,700 hours, ~500GB+).
🚀 Cloning NPTEL2020 repository for download scripts...
⏭️ NPTEL2020 repo already exists.
🚀 Information for SPIRE-SIES...
Reference: https://arxiv.org/abs/2312.00698
Note: SPIRE-SIES often requires requesting access or following the paper's data release instructions.
🚀 Information for Accent DB...
Link: https://accentdb.org/
Note: Please download the Indian accent subset manually from the website if automated scripts are unavailable.


## 4. Phase 1: Preprocessing
Standardizes audio and generates the manifest for training. Choose your dataset below.

In [9]:
# OPTION A: Preprocess Svarah (Hugging Face Dataset)
!python src/preprocess.py \
    --hf_dataset ai4bharat/Svarah \
    --output_dir data/processed/svarah \
    --manifest_path data/processed/svarah_manifest.json

# OPTION B: Preprocess IndicAccentDB (Hugging Face Dataset)
# !python src/preprocess.py \
#     --hf_dataset DarshanaS/IndicAccentDb \
#     --output_dir data/processed/indic_accent \
#     --manifest_path data/processed/indic_manifest.json

--- IndiVoice Preprocessing Engine v1.4 (torchcodec Stability) ---
Loading Hugging Face dataset: ai4bharat/Svarah...
✅ Successfully loaded 'test' split.
Using 'audio_filepath' for audio and 'text' for transcripts.
Preprocessing 6656 samples...
100% 6656/6656 [02:32<00:00, 43.71it/s]
HF Dataset processed and manifest saved to data/processed/svarah_manifest.json


## 5. Phase 2: Deep Learning Training
Fine-tunes Whisper-Medium using LoRA. Best checkpoints are saved back to your Google Drive.

In [7]:
# Double Check GPU before starting heavy training
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU NOT DETECTED. Connect to a T4 GPU runtime before training.")

!python src/train.py \
    --train_manifest data/processed/svarah_manifest.json \
    --val_manifest data/processed/svarah_manifest.json \
    --output_dir models/whisper-indian-lora

RuntimeError: GPU NOT DETECTED. Connect to a T4 GPU runtime before training.

## 6. Phase 3: Evaluation
Test the performance on unseen data.

In [ ]:
!python src/evaluate.py \
    --model_path models/whisper-indian-lora/final \
    --test_manifest data/processed/svarah_manifest.json

2026-03-13 20:36:19.914290: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773434180.130357   21210 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773434180.186223   21210 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773434180.588390   21210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773434180.588439   21210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773434180.588445   21210 computation_placer.cc:177] computation placer alr

## 7. Phase 4: Launch Demo
Starts a Gradio web app. Click the public `gradio.live` link to try your model.

In [ ]:
!python src/deploy.py --model_path models/whisper-indian-lora/final --share

2026-03-13 19:38:09.629898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773430689.650633    5381 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773430689.657537    5381 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773430689.676376    5381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773430689.676412    5381 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773430689.676424    5381 computation_placer.cc:177] computation placer alr